<a href="https://colab.research.google.com/github/2403a52208-sudo/NLP/blob/main/2403A52208_B09_assg6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
data=pd.read_excel('/content/drive/MyDrive/LDA-Data.xlsx')
data.head()

,News
0,Virat scored century in match
1,BJP won in elections
2,Bumra took 5 wicket in a match
3,Congress form state government


#Text Preprocesing: Clean Text, Word Tokenization, stopword removal, lemma,Rejoin

In [3]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

def nltk_preprocessing_pipeline(text):
    # Initialize NLTK tools
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))

    # 1. Preprocess text (from previous steps)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)

    text = text.lower()  # Convert to lowercase

    # Remove emojis
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags (iOS)
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
        )
    text = emoji_pattern.sub(r'', text)

    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces

    # 2. Word Tokenization
    tokenized_words = word_tokenize(text)

    # 3. Stopword Removal
    filtered_words = [word for word in tokenized_words if word not in stop_words]

    # 4. Lemmatization
    lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]

    # 5. Rejoin words
    clean_summary = ' '.join(lemmatized_words)

    return clean_summary

print("NLTK preprocessing pipeline function created successfully!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


NLTK preprocessing pipeline function created successfully!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [4]:
data['clean_News'] = data['News'].apply(nltk_preprocessing_pipeline)
print("\nComparison of previous clean_summaries and new clean_summaries_pipeline (first 5 rows):")
print(data[['clean_News']].head())


Comparison of previous clean_summaries and new clean_summaries_pipeline (first 5 rows):
                       clean_News
0      virat scored century match
1                    bjp election
2       bumra took 5 wicket match
3  congress form state government


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
# Vectorize the cleaned summaries
count_vectorizer = CountVectorizer(max_df=0.95, min_df=1, stop_words='english')
doc_term_matrix = count_vectorizer.fit_transform(data['clean_News'])



In [6]:
import pandas as pd
# Get feature (word) names
feature_names = count_vectorizer.get_feature_names_out()
# Convert sparse matrix to DataFrame
bow_df = pd.DataFrame(doc_term_matrix.toarray(),columns=feature_names)
# Display BoW matrix for top 10 documents
bow_top_10 = bow_df.head(10)
print(bow_top_10)

   bjp  bumra  century  congress  election  form  government  match  scored  \
0    0      0        1         0         0     0           0      1       1   
1    1      0        0         0         1     0           0      0       0   
2    0      1        0         0         0     0           0      1       0   
3    0      0        0         1         0     1           1      0       0   

   state  took  virat  wicket  
0      0     0      1       0  
1      0     0      0       0  
2      0     1      0       1  
3      1     0      0       0  


#LDA with Sample Data and BoW

In [7]:
from sklearn.decomposition import LatentDirichletAllocation

# Initialize and fit LDA model
num_topics = 2
LDA = LatentDirichletAllocation(n_components=num_topics, random_state=42)
LDA.fit(doc_term_matrix)

LatentDirichletAllocation(n_components=2, random_state=42)

In [8]:
def display_topics(model, feature_names, num_top_words):
    for topic_idx in range(len(model.components_)):
        print(f"\nTopic {topic_idx}:")

        # Get word weights for this topic
        topic_weights = model.components_[topic_idx]

        # Get indices of words sorted by weight (descending)
        sorted_indices = topic_weights.argsort()[::-1]

        # Take top N words
        top_indices = sorted_indices[:num_top_words]

        # Print top words
        for idx in top_indices:
            print(feature_names[idx], end=" ")
        print()

In [9]:
# Display top words for each topic
num_top_words = 10
print(f"\nTop {num_top_words} words per topic:")
display_topics(LDA, count_vectorizer.get_feature_names_out(), num_top_words)


Top 10 words per topic:

Topic 0:
form government congress state election bjp match wicket bumra took 

Topic 1:
match virat century scored took bumra wicket bjp election state 


In [11]:
# Assign topics to each document
document_topics = LDA.transform(doc_term_matrix)
data['topic'] = document_topics.argmax(axis=1)
print("\nDataFrame with assigned topics (first 5 rows):")
print(data[['clean_News', 'topic']].head())


DataFrame with assigned topics (first 5 rows):
                       clean_News  topic
0      virat scored century match      1
1                    bjp election      0
2       bumra took 5 wicket match      1
3  congress form state government      0


#NMF with Sample Data and BoW
NMF Import and Fit

In [12]:
from sklearn.decomposition import NMF

# Initialize NMF model
num_topics = 2
nmf_model = NMF(n_components=num_topics, random_state=42)

# Fit the NMF model to the document-term matrix
nmf_model.fit(doc_term_matrix)
print("NMF model initialized and fitted successfully.")

NMF model initialized and fitted successfully.


In [13]:
print(f"\nTop {num_top_words} words per topic (NMF):")
display_topics(nmf_model, count_vectorizer.get_feature_names_out(), num_top_words)


Top 10 words per topic (NMF):

Topic 0:
match virat took scored wicket bumra century bjp election state 

Topic 1:
state form congress government election bjp took virat wicket scored 


# Assign_NMF_Topics

In [14]:
# Assign topics to each document
document_topics = nmf_model.transform(doc_term_matrix)
data['topic'] = document_topics.argmax(axis=1)

print("\nDataFrame with assigned topics (first 5 rows):")
print(data[['clean_News', 'topic']].head())


DataFrame with assigned topics (first 5 rows):
                       clean_News  topic
0      virat scored century match      0
1                    bjp election      1
2       bumra took 5 wicket match      0
3  congress form state government      1


In [18]:
import pandas as pd
import numpy as np
df_arxiv = pd.read_csv('/content/drive/MyDrive/arxiv_data.csv')
df_arxiv.head()

,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV']
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']"


#Text Preprocesing: Clean Text, Word Tokenization, stopword removal, lemma,Rejoin

In [19]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [20]:
def nltk_preprocessing_pipeline(text):
    # Initialize NLTK tools
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))

    # 1. Preprocess text (from previous steps)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)

    text = text.lower()  # Convert to lowercase

    # Remove emojis
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags (iOS)
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)

    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces

    # 2. Word Tokenization
    tokenized_words = word_tokenize(text)

    # 3. Stopword Removal
    filtered_words = [word for word in tokenized_words if word not in stop_words]

    # 4. Lemmatization
    lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]

    # 5. Rejoin words
    clean_summary = ' '.join(lemmatized_words)

    return clean_summary

print("NLTK preprocessing pipeline function created successfully!")

NLTK preprocessing pipeline function created successfully!


In [22]:
df_arxiv['clean_summaries_pipeline'] = df_arxiv['summaries'].apply(nltk_preprocessing_pipeline)
print("\nComparison of previous clean_summaries and new clean_summaries_pipeline (first 5 rows):")
print(df_arxiv[['clean_summaries_pipeline']].head())


Comparison of previous clean_summaries and new clean_summaries_pipeline (first 5 rows):
                            clean_summaries_pipeline
0  stereo matching one widely used technique infe...
1  recent advancement artificial intelligence ai ...
2  paper proposed novel mutual consistency networ...
3  consistency training proven advanced semisuper...
4  ensure safety automated driving correct percep...


In [24]:
from sklearn.feature_extraction.text import CountVectorizer
# Vectorize the cleaned summaries
count_vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
doc_term_matrix = count_vectorizer.fit_transform(df_arxiv['clean_summaries_pipeline'])

In [25]:
import pandas as pd
# Get feature (word) names
feature_names = count_vectorizer.get_feature_names_out()
# Convert sparse matrix to DataFrame
bow_df = pd.DataFrame(doc_term_matrix.toarray(),columns=feature_names)
# Display BoW matrix for top 10 documents
bow_top_10 = bow_df.head(10)
print(bow_top_10)

   00  000  00001  00002  00005  0001  00010  0002  0003  0004  ...  zslkg  \
0   0    0      0      0      0     0      0     0     0     0  ...      0   
1   0    0      0      0      0     0      0     0     0     0  ...      0   
2   0    0      0      0      0     0      0     0     0     0  ...      0   
3   0    0      0      0      0     0      0     0     0     0  ...      0   
4   0    0      0      0      0     0      0     0     0     0  ...      0   
5   0    0      0      0      0     0      0     0     0     0  ...      0   
6   0    0      0      0      0     0      0     0     0     0  ...      0   
7   0    0      0      0      0     0      0     0     0     0  ...      0   
8   0    0      0      0      0     0      0     0     0     0  ...      0   
9   0    0      0      0      0     0      0     0     0     0  ...      0   

   zspu  zssbir  zsslr  zssr  zurich  zvos  zynq  zynq7000  zynqnet  
0     0       0      0     0       0     0     0         0        0  
1

#LDA with Kaggle Data and BoW

In [26]:
from sklearn.decomposition import LatentDirichletAllocation

# Initialize and fit LDA model
num_topics = 2
LDA = LatentDirichletAllocation(n_components=num_topics, random_state=42)
LDA.fit(doc_term_matrix)

LatentDirichletAllocation(n_components=2, random_state=42)

In [27]:
def display_topics(model, feature_names, num_top_words):
    for topic_idx in range(len(model.components_)):
        print(f"\nTopic {topic_idx}:")

        # Get word weights for this topic
        topic_weights = model.components_[topic_idx]

        # Get indices of words sorted by weight (descending)
        sorted_indices = topic_weights.argsort()[::-1]

        # Take top N words
        top_indices = sorted_indices[:num_top_words]

        # Print top words
        for idx in top_indices:
            print(feature_names[idx], end=" ")
        print()

In [28]:
# Display top words for each topic
num_top_words = 10
print(f"\nTop {num_top_words} words per topic:")
display_topics(LDA, count_vectorizer.get_feature_names_out(), num_top_words)


Top 10 words per topic:

Topic 0:
image method network object model feature detection segmentation proposed propose 

Topic 1:
learning model data method network graph algorithm task approach problem 


In [30]:
# Assign topics to each document
document_topics = LDA.transform(doc_term_matrix)
df_arxiv['topic'] = document_topics.argmax(axis=1)

print("\nDataFrame with assigned topics (first 5 rows):")
print(df_arxiv[['clean_summaries_pipeline', 'topic']].head())


DataFrame with assigned topics (first 5 rows):
                            clean_summaries_pipeline  topic
0  stereo matching one widely used technique infe...      0
1  recent advancement artificial intelligence ai ...      1
2  paper proposed novel mutual consistency networ...      0
3  consistency training proven advanced semisuper...      0
4  ensure safety automated driving correct percep...      0


#NMF with Kaggle Data and BoW

In [31]:
from sklearn.decomposition import NMF

# Initialize NMF model
num_topics = 2
nmf_model = NMF(n_components=num_topics, random_state=42)

# Fit the NMF model to the document-term matrix
nmf_model.fit(doc_term_matrix)
print("NMF model initialized and fitted successfully.")

NMF model initialized and fitted successfully.


In [32]:
num_top_words = 10
print(f"\nTop {num_top_words} words per topic (NMF):")
display_topics(nmf_model, count_vectorizer.get_feature_names_out(), num_top_words)


Top 10 words per topic (NMF):

Topic 0:
model learning data network method graph task approach representation feature 

Topic 1:
image method object network segmentation feature detection proposed result propose 


In [34]:
# Assign topics to each document
document_topics = nmf_model.transform(doc_term_matrix)
df_arxiv['topic'] = document_topics.argmax(axis=1)

print("\nDataFrame with assigned topics (first 5 rows):")
print(df_arxiv[['clean_summaries_pipeline', 'topic']].head())


DataFrame with assigned topics (first 5 rows):
                            clean_summaries_pipeline  topic
0  stereo matching one widely used technique infe...      1
1  recent advancement artificial intelligence ai ...      1
2  paper proposed novel mutual consistency networ...      0
3  consistency training proven advanced semisuper...      0
4  ensure safety automated driving correct percep...      0


#NMF with Sample Data and TFIDF

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer

**Create TF-IDF Document–Term Matrix**

In [36]:
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=1,
    stop_words='english'
)

tfidf_matrix_sample = tfidf_vectorizer.fit_transform(data['clean_News'])

Fit NMF on TF-IDF Matrix

In [37]:
from sklearn.decomposition import NMF

num_topics = 2
nmf_tfidf_sample = NMF(n_components=num_topics, random_state=42)
nmf_tfidf_sample.fit(tfidf_matrix_sample)

print("NMF with TF-IDF (Sample Data) fitted successfully.")

NMF with TF-IDF (Sample Data) fitted successfully.


#Display Topics

In [38]:
num_top_words = 10
print(f"\nTop {num_top_words} words per topic (NMF + TF-IDF | Sample Data):")
display_topics(
    nmf_tfidf_sample,
    tfidf_vectorizer.get_feature_names_out(),
    num_top_words
)


Top 10 words per topic (NMF + TF-IDF | Sample Data):

Topic 0:
match bumra wicket took virat scored century state government election 

Topic 1:
election bjp form government state congress took virat wicket scored 


Assign Topics to Documents

In [39]:
sample_doc_topics = nmf_tfidf_sample.transform(tfidf_matrix_sample)
data['tfidf_nmf_topic'] = sample_doc_topics.argmax(axis=1)

print(data[['clean_News', 'tfidf_nmf_topic']])

                       clean_News  tfidf_nmf_topic
0      virat scored century match                0
1                    bjp election                1
2       bumra took 5 wicket match                0
3  congress form state government                1


#NMF with Kaggle Data and TF-IDF
Create TF-IDF Document–Term Matrix

In [41]:
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=2,
    stop_words='english'
)

tfidf_matrix_kaggle = tfidf_vectorizer.fit_transform(df_arxiv['clean_summaries_pipeline'])

In [42]:
num_topics = 2
nmf_tfidf_kaggle = NMF(n_components=num_topics, random_state=42)
nmf_tfidf_kaggle.fit(tfidf_matrix_kaggle)

print("NMF with TF-IDF (Kaggle Data) fitted successfully.")

NMF with TF-IDF (Kaggle Data) fitted successfully.


#Display Topics

In [43]:
num_top_words = 10
print(f"\nTop {num_top_words} words per topic (NMF + TF-IDF | Kaggle Data):")
display_topics(
    nmf_tfidf_kaggle,
    tfidf_vectorizer.get_feature_names_out(),
    num_top_words
)


Top 10 words per topic (NMF + TF-IDF | Kaggle Data):

Topic 0:
image model method learning object network data feature task segmentation 

Topic 1:
graph node representation network gnns learning neural model structure data 


#Assign Topics to Documents

In [45]:
kaggle_doc_topics = nmf_tfidf_kaggle.transform(tfidf_matrix_kaggle)
df_arxiv['tfidf_nmf_topic'] = kaggle_doc_topics.argmax(axis=1)

print(df_arxiv[['clean_summaries_pipeline', 'tfidf_nmf_topic']].head())

                            clean_summaries_pipeline  tfidf_nmf_topic
0  stereo matching one widely used technique infe...                0
1  recent advancement artificial intelligence ai ...                0
2  paper proposed novel mutual consistency networ...                0
3  consistency training proven advanced semisuper...                0
4  ensure safety automated driving correct percep...                0
